# 異種出力モデルによる制約付きベイズ最適化：回帰GP＋二値分類GP

**強度を最大化**しながら、**導電率の下限**と**合否の二値制約**を考慮する例です。回帰GPと分類GPを `HeterogeneousModel` にまとめ、既存の `ProblemSemantics` と `make_mixed_constrained_qei_acquisition` で候補を提案します。

学習 → 候補提案 → 強度・導電率・合否の観測 → 両モデルの再学習 → 次の候補提案まで扱います。異種**出力**のモデルであり、入力特徴量がタスクごとに異なる `HeterogeneousMTGP` とは別です。


In [ ]:
import torch
from botorch.sampling.normal import SobolQMCNormalSampler

from robotorchan.acquisition.composition import make_mixed_constrained_qei_acquisition
from robotorchan.benchmarks.heterogeneous_synthetic import evaluate_truth, observe
from robotorchan.models.classification import BinarySingleTaskGPClassifier
from robotorchan.models.heterogeneous import HeterogeneousModel
from robotorchan.models.standard.single_task import SingleTaskGP
from robotorchan.optim.dispatch import optimize_acqf
from robotorchan.semantics import (
    ClassificationConstraint,
    ContinuousConstraint,
    ProblemSemantics,
    RegressionObjective,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(1212)


## 問題設定：3つの出力の役割

| 出力名 | 種類 | 目的 |
|---|---|---|
| `strength` | 連続値・回帰 | **最大化する目的関数** |
| `conductivity` | 連続値・回帰 | **下限を満たす制約** |
| `pass` | 0/1ラベル・二値分類 | **合格クラスに関する制約** |

分類モデルが予測するのは合格ラベルの**確率**です。潜在GPの平均を合格確率とみなしてはいけません。

## 1. 観測データとモデルの学習

強度・導電率は連続値なので回帰GP、合否は0/1の観測ラベルなのでBernoulli尤度を持つ二値分類GPを使います。

それぞれのモデルの既存 `fit()` を利用します。回帰はExact GPの周辺尤度、分類は変分GPのELBOを最適化しますが、Notebook内で独自の学習ループを実装する必要はありません。


### 初期観測データの意味

`observe(train_X, seed=...)` は合成実験の観測値を返します。`strength` と `conductivity` は連続値、`passed` は実際に得られた0/1ラベルです。

回帰モデルには既知の観測ノイズ分散を指定し、分類モデルには観測ラベルを渡します。分類の教師データに真の合格確率を渡さないことが重要です。

In [ ]:
generator = torch.Generator().manual_seed(1212)
train_X = torch.rand(24, 3, generator=generator)
observed = observe(train_X, seed=2212)
train_Y = torch.stack((observed.strength, observed.conductivity), dim=-1)

regression = SingleTaskGP(
    train_X,
    train_Y,
    train_Yvar=torch.full_like(train_Y, 0.02**2),
)
regression.fit()
regression.eval()

classifier = BinarySingleTaskGPClassifier(
    train_X,
    observed.passed,
    inducing_points=8,
)
classifier.fit(epochs=12, lr=0.03)


### モデルごとの予測を確認する

分類の `predict_proba` は `[P(pass=0), P(pass=1)]` を返します。回帰GPの事後平均と並べて確認しますが、両者は**異なる意味・単位**を持つため、そのまま数値を比較して最適化の目的にしません。

In [ ]:
probe_X = torch.tensor([[0.25, 0.50, 0.75], [0.75, 0.50, 0.25]])
with torch.no_grad():
    regression_mean = regression.posterior(probe_X).mean
    pass_probabilities = classifier.predict_proba(probe_X)
print("Regression posterior means [strength, conductivity]:\n", regression_mean)
print("Classification probabilities [P(fail), P(pass)]:\n", pass_probabilities)
assert regression_mean.shape == (2, 2)
assert pass_probabilities.shape == (2, 2)
torch.testing.assert_close(pass_probabilities.sum(dim=-1), torch.ones(2), atol=1e-5, rtol=1e-5)

## 2. 出力の意味付けと制約付き獲得関数

強度を最大化し、導電率の下限と合格クラスの確率を組み合わせます。分類のlatent GP平均を合格確率として扱いません。


### 名前付き出力と制約の意味

`HeterogeneousModel` は回帰と分類の異なる予測機構を名前付き出力として扱います。`ProblemSemantics` では、`strength` を最大化し、`conductivity` に下限、`pass` に合格クラスの制約を指定します。

`make_mixed_constrained_qei_acquisition` は**既存の合成獲得関数API**です。Notebook内で回帰値と分類の潜在平均を手作業で掛け合わせる処理は追加しません。

`best_f` は獲得関数の改善基準です。可行観測点を基準とする設計が必要な場合は、既存の制約セマンティクスと観測可行性の定義を確認して指定します。

In [ ]:
model = HeterogeneousModel(
    regression,
    classifier,
    output_names=["strength", "conductivity", "pass"],
)
semantics = ProblemSemantics(
    objectives=(RegressionObjective("strength"),),
    constraints=(
        ContinuousConstraint("conductivity", threshold=0.8),
        ClassificationConstraint("pass", feasible_class=1),
    ),
)
acquisition = make_mixed_constrained_qei_acquisition(
    model,
    semantics,
    best_f=train_Y[:, 0].max().item(),
    sampler=SobolQMCNormalSampler(torch.Size([16]), seed=1212),
)


### 制約付き獲得関数は何を評価するか

強度が高くても導電率や合否条件を満たせなければ望ましい候補ではありません。獲得関数は、回帰と分類それぞれの予測に基づき、改善と制約を組み合わせて候補を評価します。

二値分類の合格確率は、潜在関数の事後平均ではなく、分類尤度を通じて得られる予測確率です。

## 3. 候補の提案と実際の評価

非微分的な構成も扱える有限Sobol探索を使います。optimizer APIの互換性そのものは別の監査フローで扱います。


### Sobol探索で候補を提案する

異種モデルの獲得関数には、勾配が利用しにくい構成が含まれることがあります。ここでは既存の `optimize_acqf(optimizer="sobol")` を使い、探索範囲内の候補を評価します。

`score` は獲得関数値であり、測定した強度や合格確率そのものではありません。

In [ ]:
bounds = torch.stack((torch.zeros(3), torch.ones(3)))
candidate, score = optimize_acqf(
    acquisition,
    bounds,
    q=1,
    optimizer="sobol",
    raw_samples=32,
    seed=1213,
)
with torch.no_grad():
    strength, conductivity, pass_probability = evaluate_truth(candidate)
    predicted_pass_probability = model.entry_predict_proba(1, candidate)[..., 1]

assert candidate.shape == torch.Size([1, 3])
assert torch.isfinite(score).all()
assert torch.isfinite(strength).all()
assert torch.isfinite(conductivity).all()
assert torch.isfinite(pass_probability).all()
print("candidate:", candidate)
print("acquisition score:", score)
print("strength:", strength)
print("conductivity:", conductivity)
print("true pass probability:", pass_probability)
print("predicted pass probability:", predicted_pass_probability)


### 合格確率と実際の合否は異なる

`evaluate_truth` の `pass_probability` は合成問題で定義された**真の合格確率**です。現実の実験では通常この値を直接観測できず、合格または不合格という**0/1の結果**を得ます。

次のセルではBernoulli分布から0/1ラベルを1件生成し、分類器の追加学習に利用します。

## 4. 観測とモデル予測の区別

`evaluate_truth` の合格確率は合成問題の真値であり、実験で観測される二値ラベルとは異なります。分類器を再学習する場合は、新しい観測ラベルを取得してから学習データへ追加してください。ここでは候補の評価までを示します。


## 5. 二値観測を追加して再学習

合格確率は直接の二値観測ではありません。この合成例では真の合格確率からBernoulli観測を1件生成します。回帰出力と分類ラベルを追加し、両モデルを再学習します。


### 回帰と分類の両方を再学習する

候補の観測後には、強度・導電率・合否ラベルを同じ入力 `candidate` に対応させて訓練データへ追加します。

`SingleTaskGP.fit()` と `BinarySingleTaskGPClassifier.fit()` を再利用し、各モデルの学習目的に応じた処理をライブラリ側へ委ねます。

### 予測確率と観測ラベルの対応

合格確率が0.8でも、実験結果が必ず合格になるわけではありません。`torch.bernoulli` は合格確率に従って0/1をサンプリングします。

**分類モデルへ追加する教師信号は0/1ラベル**であり、確率を連続回帰のターゲットとして扱いません。

In [ ]:
label_generator = torch.Generator().manual_seed(1214)
observed_pass = torch.bernoulli(
    pass_probability.detach(), generator=label_generator
).long()
updated_X = torch.cat([train_X, candidate.detach()], dim=0)
updated_Y = torch.cat(
    [train_Y, torch.stack((strength, conductivity), dim=-1).detach()],
    dim=0,
)
updated_passed = torch.cat([observed.passed, observed_pass], dim=0)

updated_regression = SingleTaskGP(
    updated_X,
    updated_Y,
    train_Yvar=torch.full_like(updated_Y, 0.02**2),
)
updated_regression.fit()
updated_regression.eval()

updated_classifier = BinarySingleTaskGPClassifier(
    updated_X,
    updated_passed,
    inducing_points=8,
)
updated_classifier.fit(epochs=12, lr=0.03)

updated_model = HeterogeneousModel(
    updated_regression,
    updated_classifier,
    output_names=["strength", "conductivity", "pass"],
)
semantics.validate(updated_model)
assert updated_X.shape[0] == train_X.shape[0] + 1
assert updated_passed.shape[0] == observed.passed.shape[0] + 1
print("new observed label:", observed_pass)
print("updated observations:", updated_X.shape[0])


## 6. 再学習した異種モデルで次の候補を提案する

更新した回帰GP・分類GPを `HeterogeneousModel` にまとめ直し、同じ `ProblemSemantics` から獲得関数を再構築します。新しい強度観測を踏まえた改善基準を使います。

次の候補はまだ実験していないため、**予測確率と観測ラベルを混同しません**。

In [ ]:
updated_acquisition = make_mixed_constrained_qei_acquisition(
    updated_model,
    semantics,
    best_f=updated_Y[:, 0].max().item(),
    sampler=SobolQMCNormalSampler(torch.Size([16]), seed=1215),
)
next_candidate, next_score = optimize_acqf(
    updated_acquisition,
    bounds,
    q=1,
    optimizer="sobol",
    raw_samples=32,
    seed=1216,
)
with torch.no_grad():
    next_pass_probability = updated_model.entry_predict_proba(1, next_candidate)[..., 1]
print("Next candidate:", next_candidate)
print("Next acquisition score:", next_score)
print("Predicted pass probability:", next_pass_probability)
assert next_candidate.shape == (1, 3)
assert torch.isfinite(next_score).all()
assert torch.isfinite(next_pass_probability).all()

## 7. 観測前後の予測を比較する

観測した候補における**強度と導電率の事後平均**、および**合格クラスの予測確率**を比較します。分類確率の変化と回帰平均の変化は異なる量です。

In [ ]:
with torch.no_grad():
    before_regression = regression.posterior(candidate).mean.squeeze(0)
    after_regression = updated_regression.posterior(candidate).mean.squeeze(0)
    before_pass = classifier.predict_proba(candidate)[..., 1]
    after_pass = updated_classifier.predict_proba(candidate)[..., 1]
print("Regression mean before:", before_regression)
print("Regression mean after:", after_regression)
print("Pass probability before:", before_pass)
print("Pass probability after:", after_pass)
assert before_regression.shape == after_regression.shape
assert torch.isfinite(after_pass).all()

## 8. 公開APIと責務の整理

| 役割 | 既存API | 注意点 |
|---|---|---|
| 強度・導電率の回帰 | `SingleTaskGP.fit()` | 連続値の2出力 |
| 合否の分類 | `BinarySingleTaskGPClassifier.fit()` | 0/1ラベル、変分ELBO |
| 合格確率 | `predict_proba()` | 潜在GP平均とは異なる |
| 異種出力の統合 | `HeterogeneousModel` | 名前付き出力 |
| 目的・制約の定義 | `ProblemSemantics` | 回帰目的・連続制約・分類制約 |
| 制約付き獲得関数 | `make_mixed_constrained_qei_acquisition` | 既存の合成API |
| 候補提案 | `optimize_acqf(optimizer="sobol")` | q=1の有限Sobol探索 |

**重要：** 合成問題の真の合格確率は、通常の実験で得られる0/1ラベルとは異なります。再学習には実際に観測したラベルを用います。